In [86]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import TargetEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline

In [87]:
train = pd.read_csv('clean-data/sold_train.csv')

/var/folders/t2/p9112v_n469068__fty8_3nc0000gn/T/ipykernel_23704/3567145975.py:1: DtypeWarning: Columns (7,61,62) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('clean-data/sold_train.csv')


## Setup: Cross-Validation
Implement k-fold cross-validation on the training set to estimate the test error of each model before final evaluation

In [97]:
# helper function that performs k-fold cross-validation on a specific subset of features
def evaluate_feature_subset(train_set, features, target, n_splits=10, random_state=715, log=False):
    """
    parameters:
    ----------
    train_set: training dataset containing all features and target
    features: names of feature columns (list of strings)
    target: name of target column (string)
    n_splits: number of folds for cross-validation
    random_state: seed for reproducible shuffling

    return:
    -------
    val_results: dictionary containing the mean and standard deviation of RMSE and R2

    """
    # keep features as a dataframe so that ColumnTransformer can look up data type
    X = train_set[features]
    y = train_set[target].to_numpy()

    # identify categorical versus numeric features from the subset
    categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
    categorical_features = list(set(categorical_features)) # defensive step to deduplicate features

    numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
    numeric_features = list(set(numeric_features))

    # handle missing values across categorical features and encode the features
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
        ('encoder', TargetEncoder(cv=5, random_state=random_state)) # no need to specify cv because KFold will handle the cross-validation
    ])

    # apply the pipeline to categorical features and leave numeric features untouched
    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', categorical_transformer, categorical_features), 
            ('num', 'passthrough', numeric_features)
            ],
        remainder='drop'
    )

    # shuffle the training set to break order dependencies dictated by close date
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_rmse = []
    fold_r2 = []

    # execute the cross-validation loop, storing the validation results from each fold
    for train_i, val_i in kf.split(X):
        X_train, X_val = X.iloc[train_i], X.iloc[val_i] # recall that features stay in a dataframe
        y_train, y_val = y[train_i], y[val_i]           # while target has been converted to an array
        
        # preprocess categorical features before fitting the model
        base_pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('regressor', LinearRegression())
        ])
        
        if log:
            model_pipeline = TransformedTargetRegressor(
                regressor=base_pipeline,
                func=np.log1p,
                inverse_func=np.expm1
            )
        else:
            model_pipeline = base_pipeline
        
        model_pipeline.fit(X_train, y_train)
        y_pred = model_pipeline.predict(X_val)

        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        r2 = r2_score(y_val, y_pred)
        fold_rmse.append(rmse)
        fold_r2.append(r2)
    
    # compute the mean and standard deviation of rmse and r2
    val_results = {
        'subset': features, 
        'mean_rmse': np.mean(fold_rmse),
        'sd_rmse': np.std(fold_rmse),
        'mean_r2': np.mean(fold_r2),
        'sd_r2': np.std(fold_r2)
    }
    return val_results

## Baseline
Fit a linear regression model without feature engineering or rescaling

In [89]:
# key features include living area, days on market, 
# county or parish, city, postal code, latitude, longitude
# lot size, bedrooms, bathrooms
# year built, listing contract date, purchase contract date, close date
train.columns

Index(['BuyerAgentAOR', 'ListAgentAOR', 'Flooring', 'ViewYN', 'PoolPrivateYN',
       'OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate',
       'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude',
       'Longitude', 'UnparsedAddress', 'LivingArea', 'ListPrice',
       'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName',
       'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName',
       'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName',
       'AssociationFeeFrequency', 'ListingKeyNumeric', 'MLSAreaMajor',
       'CountyOrParish', 'MlsStatus', 'ElementarySchool', 'AttachedGarageYN',
       'ParkingTotal', 'LotSizeAcres', 'SubdivisionName', 'BuyerOfficeAOR',
       'YearBuilt', 'StreetNumberNumeric', 'ListingId',
       'BathroomsTotalInteger', 'City', 'BedroomsTotal',
       'ContractStatusChangeDate', 'PurchaseContractDate',
       'ListingContractDate', 'StateOrProvince', 'MiddleOrJuniorSchool',
       'Fi

In [98]:
# large mean RMSE ($6.8 million) and very poor mean R2 (less than 10% variation explained)
target = ['ClosePrice']
baseline_features = ['LivingArea', 'DaysOnMarket', 'LotSizeSquareFeet', 'BedroomsTotal', 'BathroomsTotalInteger']
evaluate_feature_subset(train, baseline_features, target)

{'subset': ['LivingArea',
  'DaysOnMarket',
  'LotSizeSquareFeet',
  'BedroomsTotal',
  'BathroomsTotalInteger'],
 'mean_rmse': np.float64(6784153.057970678),
 'sd_rmse': np.float64(3706708.8161766003),
 'mean_r2': np.float64(0.09841661310450323),
 'sd_r2': np.float64(0.16188008471478282)}

Incorporate some location details and refit the model

In [99]:
# mean R2 increased by nearly 4% but mean RMSE is still large
baseline_features = baseline_features + ['City']
evaluate_feature_subset(train, baseline_features, target)

/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/wenxincui/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed w

{'subset': ['LivingArea',
  'DaysOnMarket',
  'LotSizeSquareFeet',
  'BedroomsTotal',
  'BathroomsTotalInteger',
  'City'],
 'mean_rmse': np.float64(6729385.555706108),
 'sd_rmse': np.float64(3755444.2944225236),
 'mean_r2': np.float64(0.13753025690116685),
 'sd_r2': np.float64(0.225716627851214)}

## Phase 1: Rescale Numeric Features
Scale numeric features using the median and IQR to prevent variables with larger ranges from dominating the model and to reduce the influence of extreme property values

In [100]:
from sklearn.preprocessing import RobustScaler
target = 'ClosePrice'
numeric_features = ['LivingArea', 'DaysOnMarket', 'LotSizeSquareFeet', 'BedroomsTotal', 'BathroomsTotalInteger']

# NOTE: use the scaler to transform the features in test set
scaler = RobustScaler()
train_rescaled = train.copy()
train_rescaled[numeric_features] = scaler.fit_transform(train[numeric_features])

In [101]:
# no improvement (as expected) because scaling only affects coefficients
phase1_features = ['LivingArea', 'DaysOnMarket', 'LotSizeSquareFeet', 'BedroomsTotal', 
                   'BathroomsTotalInteger', 'City']
evaluate_feature_subset(train_rescaled, phase1_features, target)

{'subset': ['LivingArea',
  'DaysOnMarket',
  'LotSizeSquareFeet',
  'BedroomsTotal',
  'BathroomsTotalInteger',
  'City'],
 'mean_rmse': np.float64(6729385.555706108),
 'sd_rmse': np.float64(3755444.294422522),
 'mean_r2': np.float64(0.13753025690116633),
 'sd_r2': np.float64(0.22571662785121172)}

Adopt a log-log model that log transforms right-skewed target and numeric features

In [102]:
train_log = train.copy()
log_features = ['LivingArea', 'DaysOnMarket', 'BathroomsTotalInteger']
for feature in log_features:
    train_log[feature] = np.log1p(train_log[feature])

In [103]:
# make sure there are no nulls after log transformation
train_log[phase1_features].isna().sum()

LivingArea                0
DaysOnMarket              0
LotSizeSquareFeet         0
BedroomsTotal             0
BathroomsTotalInteger     0
City                     39
dtype: int64

In [ ]:
# mean R2 improved by 1% but showed a larger variation
# mean RMSE decreased slightly
evaluate_feature_subset(train_log, phase1_features, target, log=True)

{'subset': ['LivingArea',
  'DaysOnMarket',
  'LotSizeSquareFeet',
  'BedroomsTotal',
  'BathroomsTotalInteger',
  'City'],
 'mean_rmse': np.float64(6720750.269285257),
 'sd_rmse': np.float64(3772059.693648191),
 'mean_r2': np.float64(0.14568068467107875),
 'sd_r2': np.float64(0.24218891263817707)}